In [34]:
from carculator import *
cip = CarInputParameters()
cip.static()
dcts, array = fill_xarray_from_input_parameters(cip)

# if "country" in array.coords:
#     array = array.sel(country="NO")
# elif "region" in array.coords:
#     array = array.sel(region="NO")

import numpy as np
years = np.arange(2000, 2051)
array_new = array.interp(year=years)

# kwargs={"fill_value": "extrapolate"}

cm = CarModel(array_new , country='NO', cycle="WLTC")
cm.set_all()


# cm = CarModel(array)
# cm.set_all()

C:\Users\jansn\Anaconda3\envs\carculator\lib\site-packages\carculator_utils\array.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols] = df[cols].apply(lambda x: x.ffill())


In [21]:
import pandas as pd
import numpy as np

# 1) Flatten to a DataFrame so we can search values
df = array.to_dataframe().reset_index()   # cols: size, powertrain, parameter, year, value

# 2) See if there are any string values (likely where country lives)
df_str = df[df["value"].apply(lambda x: isinstance(x, (str, bytes)))]
print("Unique string values found:", sorted(df_str["value"].unique())[:20])

Unique string values found: []


In [11]:
# Look at the structure
print(cm.array)  
print(cm.array.coords)

car = cm.array.sel(year=2030, size="Medium", powertrain="BEV")

print(car.parameter)            # overview of all parameters for this car
# print(car.to_dataframe().T)  # pretty tabular form

<xarray.DataArray 'data' (size: 9, powertrain: 9, parameter: 358, year: 1,
                          value: 1)> Size: 232kB
array([[[[[0.00000000e+00]],

         [[0.00000000e+00]],

         [[0.00000000e+00]],

         ...,

         [[1.40000002e+02]],

         [[4.41445415e-01]],

         [[8.50000000e-01]]],


        [[[0.00000000e+00]],

         [[0.00000000e+00]],

         [[0.00000000e+00]],
...
         [[1.40000002e+02]],

         [[3.13088359e-01]],

         [[8.07031523e-01]]],


        [[[0.00000000e+00]],

         [[0.00000000e+00]],

         [[0.00000000e+00]],

         ...,

         [[1.40000002e+02]],

         [[3.53667401e-01]],

         [[8.07031523e-01]]]]])
Coordinates:
  * size        (size) object 72B 'Large' 'Large SUV' ... 'Small' 'Van'
  * powertrain  (powertrain) object 72B 'BEV' 'FCEV' ... 'PHEV-d' 'PHEV-p'
  * parameter   (parameter) object 3kB '1-Pentene direct emissions, rural' .....
  * value       (value) float64 8B 0.0
  * year        (

KeyError: "not all values found in index 'year'. Try setting the `method` keyword argument (example: method='nearest')."

In [3]:
import matplotlib

background_configuration = {
   'country of use' : 'NO' 
}

scope = {
    'powertrain':['BEV', 'PHEV-d'],
}

# ic = InventoryCar(cm, background_configuration = background_configuration,  scope=scope, scenario="SSP2-Baseline") # If the argument `method` is left unspecified, ReCiPe is used. It can also be specified epxlicitly.
# if `method_type` is left unspecified, midpoint indicators are use. Otherwise, specifiy endpoint.

# “SSP2-Baseline”, “SSP2-PkBudg1150” or “SSP2-PkBudg500”

# ic = InventoryCar(cm, background_configuration = background_configuration, scenario='SSP2-NPi')

# ic = InventoryCar(cm, background_configuration = background_configuration, scenario='SSP2-PkBudg500')

ic = InventoryCar(cm, background_configuration = background_configuration, scenario='SSP2-PkBudg500', method="recipe", indicator="midpoint")

results = ic.calculate_impacts()

results.sel(impact_category='climate change', size='Large', value=0)\
    .to_dataframe('impact')\
    .unstack(level=2)['impact']\
    .plot(kind='bar',
                stacked=True,
         figsize=(15,10))
plt.ylabel('kg CO2-eq./vkm')
plt.show()

NameError: name 'cm' is not defined

In [28]:
print(ic)

In [ ]:
print(cip.parameters[:-1]) 

In [ ]:
print(cip.scen)

In [50]:
import pandas as pd

res = results.to_dataset(name="test")

df = res.to_dataframe().reset_index()
print(df.head())

# Or Excel
df.to_csv("carculator_results_SSP2-SSP2-PkBudg500_NOR_WLTC_endpoint.csv", index=False)


              impact_category   size powertrain  year          impact  value  \
0  acidification: terrestrial  Large        BEV  2000          glider      0   
1  acidification: terrestrial  Large        BEV  2000      powertrain      0   
2  acidification: terrestrial  Large        BEV  2000  energy storage      0   
3  acidification: terrestrial  Large        BEV  2000    energy chain      0   
4  acidification: terrestrial  Large        BEV  2000     maintenance      0   

   test  
0   0.0  
1   0.0  
2   0.0  
3   0.0  
4   0.0  


In [8]:
print("coords:", list(array.coords))

coords: ['size', 'powertrain', 'parameter', 'value', 'year']


In [15]:
print("data vars:", list(array.data_vars))

AttributeError: 'DataArray' object has no attribute 'data_vars'

In [32]:
print("cm.country:", cm.country)           # should be "NO"
print("ecm country:", getattr(cm, "ecm").country)  # built by set_all(), should be "NO"


cm.country: NO
ecm country: NO


In [33]:
def run(country):
    cm = CarModel(array, cycle="WLTC", country=country)
    cm.set_all()
    ic = InventoryCar(cm)
    r = ic.calculate_impacts() if hasattr(ic,"calculate_impacts") else ic.calculate()
    return r.to_dataframe().reset_index()

df_CH = run("CH"); df_CH["country"]="CH"
df_NO = run("NO"); df_NO["country"]="NO"

# Compare climate change impacts for BEV Medium across years
cc = (pd.concat([df_CH, df_NO])
        .query("impact_category == 'climate change' and powertrain == 'BEV' and size == 'Medium'")
        .pivot_table(index="year", columns="country", values="impact"))
print(cc.head())

KeyboardInterrupt: 

In [42]:
# Extract the lightweighting parameter for a given size and powertrain
lw = array.sel(
    parameter="lightweighting",
    powertrain="BEV",
    size="Medium"
).to_pandas()

print(lw.head(15))   # first years
print(lw.tail(15))   # last years

value    0.0
year        
2000   0.100
2010   0.100
2020   0.140
2030   0.145
2040   0.150
2050   0.150
value    0.0
year        
2000   0.100
2010   0.100
2020   0.140
2030   0.145
2040   0.150
2050   0.150


In [46]:
ic.impact_categories

{'acidification: terrestrial': {'method': 'endpoint',
  'category': 'ReCiPe 2016 v1.03, endpoint (H)',
  'type': 'acidification: terrestrial',
  'abbreviation': 'acidification: terrestrial',
  'unit': '',
  'source': 'species.yr'},
 'climate change: freshwater ecosystems': {'method': 'endpoint',
  'category': 'ReCiPe 2016 v1.03, endpoint (H)',
  'type': 'climate change: freshwater ecosystems',
  'abbreviation': 'climate change: freshwater ecosystems',
  'unit': '',
  'source': 'species.yr'},
 'climate change: terrestrial ecosystems': {'method': 'endpoint',
  'category': 'ReCiPe 2016 v1.03, endpoint (H)',
  'type': 'climate change: terrestrial ecosystems',
  'abbreviation': 'climate change: terrestrial ecosystems',
  'unit': '',
  'source': 'species.yr'},
 'ecotoxicity: freshwater': {'method': 'endpoint',
  'category': 'ReCiPe 2016 v1.03, endpoint (H)',
  'type': 'ecotoxicity: freshwater',
  'abbreviation': 'ecotoxicity: freshwater',
  'unit': '',
  'source': 'species.yr'},
 'ecotoxicit

In [2]:
from carculator import *
cip = CarInputParameters()
cip.static()
dcts, array = fill_xarray_from_input_parameters(cip)

print("Available sizes:", array.coords["size"].values)

C:\Users\jansn\Anaconda3\envs\carculator\lib\site-packages\carculator_utils\array.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols] = df[cols].apply(lambda x: x.ffill())


Available sizes: ['Large' 'Large SUV' 'Lower medium' 'Medium' 'Medium SUV' 'Micro' 'Mini'
 'Small' 'Van']
